In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score,  confusion_matrix, classification_report

df= pd.read_csv("../data/raw/StudentsPerformance.csv")
df.columns = df.columns.str.replace(' ', '_').str.replace('/', '_')
print(df.head())

   gender race_ethnicity parental_level_of_education         lunch  \
0  female        group B           bachelor's degree      standard   
1  female        group C                some college      standard   
2  female        group B             master's degree      standard   
3    male        group A          associate's degree  free/reduced   
4    male        group C                some college      standard   

  test_preparation_course  math_score  reading_score  writing_score  
0                    none          72             72             74  
1               completed          69             90             88  
2                    none          90             95             93  
3                    none          47             57             44  
4                    none          76             78             75  


In [33]:
df["Average_Score"] = df[['math_score', 'reading_score', 'writing_score']].mean(axis=1)
df['at_risk']=  (df['Average_Score'] < 60).astype(int)
print(df['at_risk'].value_counts())
print(f"\nAt-risk percentage: {df['at_risk'].mean()*100:.2f}%")



at_risk
0    715
1    285
Name: count, dtype: int64

At-risk percentage: 28.50%


In [34]:
df_model = df.copy()
catergorical_cols = ['gender','race_ethnicity','parental_level_of_education','lunch','test_preparation_course']
le_dict ={}
for col in catergorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    le_dict[col] = le
df_model.head()

,gender,race_ethnicity,parental_level_of_education,lunch,test_preparation_course,math_score,reading_score,writing_score,Average_Score,at_risk
0,0,1,1,1,1,72,72,74,72.666667,0
1,0,2,4,1,0,69,90,88,82.333333,0
2,0,1,3,1,1,90,95,93,92.666667,0
3,1,0,0,0,1,47,57,44,49.333333,1
4,1,2,4,1,1,76,78,75,76.333333,0


In [35]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

print("Before SMOTE:", y_train.value_counts().to_dict())
print("After SMOTE:", pd.Series(y_train_sm).value_counts().to_dict())

Before SMOTE: {0: 776, 1: 24}
After SMOTE: {0: 776, 1: 776}


In [29]:
feature_cols = ['gender','race_ethnicity', 'parental_level_of_education','lunch','test_preparation_course']
x = df_model[feature_cols]
y = df_model['at_risk']
X_train, X_test, y_train, y_test = train_test_split(x,y,test_size= 0.2, random_state=42, stratify=y)
print("Training Shape:",X_train.shape)
print("Testing Shape:",X_test.shape)
print("Train at-risk %:", y_train.mean()*100)
print("Test at_risk %:", y_test.mean()*100)


Training Shape: (800, 5)
Testing Shape: (200, 5)
Train at-risk %: 3.0
Test at_risk %: 3.0


In [36]:
print(X_train.columns.tolist())
print(X_train.shape)

['gender', 'race_ethnicity', 'parental_level_of_education', 'lunch', 'test_preparation_course']
(800, 5)


In [39]:
models={
    'Logisitic Regression': LogisticRegression(max_iter=1000, random_state=42,class_weight='balanced'),
    'Random Forest' : RandomForestClassifier(n_estimators=100, random_state=42,class_weight='balanced'),
    'XGBoost': XGBClassifier(random_state = 42,eval_metric = 'logloss',scale_pos_weight = len(y_train[y_train==0])/len(y_train[y_train==1]) )
}
results ={}
for name,model in models.items():
    model.fit(X_train,y_train)
    y_pred = model.predict(X_test)

    results[name] ={
        "Accuracy": accuracy_score(y_test,y_pred),
        "Precision": precision_score(y_test,y_pred,zero_division=0),
        "Recall": recall_score(y_test,y_pred,zero_division=0),
        "F1-Score": f1_score(y_test,y_pred,zero_division=0)
    }
    print(f"\n---{name}---")
    print(classification_report(y_test,y_pred,zero_division=0))
results_df = pd.DataFrame(results).T
print(results_df)


---Logisitic Regression---
              precision    recall  f1-score   support

           0       0.99      0.81      0.89       194
           1       0.10      0.67      0.17         6

    accuracy                           0.81       200
   macro avg       0.54      0.74      0.53       200
weighted avg       0.96      0.81      0.87       200


---Random Forest---
              precision    recall  f1-score   support

           0       0.97      0.86      0.91       194
           1       0.04      0.17      0.06         6

    accuracy                           0.84       200
   macro avg       0.50      0.51      0.49       200
weighted avg       0.94      0.84      0.89       200


---XGBoost---
              precision    recall  f1-score   support

           0       0.97      0.86      0.91       194
           1       0.04      0.17      0.06         6

    accuracy                           0.84       200
   macro avg       0.50      0.51      0.49       200
weighted a

In [25]:
# Instead of default 0.5 threshold, lower it so model flags more students as at-risk
from sklearn.metrics import f1_score, recall_score

threshold = 0.3  # flag as at-risk if model is 30%+ confident

print("\n=== Results with Lowered Threshold (0.3) ===")
for name, model in models.items():
    y_prob = model.predict_proba(X_test)[:, 1]
    y_pred_thresh = (y_prob >= threshold).astype(int)
    
    print(f"\n--- {name} ---")
    print(classification_report(y_test, y_pred_thresh, zero_division=0))


=== Results with Lowered Threshold (0.3) ===

--- Logisitic Regression ---
              precision    recall  f1-score   support

           0       0.97      1.00      0.98       194
           1       0.00      0.00      0.00         6

    accuracy                           0.97       200
   macro avg       0.48      0.50      0.49       200
weighted avg       0.94      0.97      0.96       200


--- Random Forest ---
              precision    recall  f1-score   support

           0       0.97      0.98      0.98       194
           1       0.00      0.00      0.00         6

    accuracy                           0.95       200
   macro avg       0.48      0.49      0.49       200
weighted avg       0.94      0.95      0.95       200


--- XGBoost ---
              precision    recall  f1-score   support

           0       0.97      0.99      0.98       194
           1       0.00      0.00      0.00         6

    accuracy                           0.96       200
   macro avg

In [ ]:
import shap
import numpy as np

# Use the best model (Logistic Regression based on recall)
best_model_name = 'Logistic Regression'
best_model_name = 'Logisitic Regression'
best_model = models[best_model_name]

# Convert to numpy array to avoid any dataframe shape issues
X_train_arr = np.array(X_train)
X_test_arr = np.array(X_test)

# LinearExplainer for Logistic Regression
explainer = shap.LinearExplainer(best_model, X_train_arr)
shap_values = explainer.shap_values(X_test_arr)

# Summary plot
shap.summary_plot(shap_values, X_test_arr, feature_names=feature_cols)


KeyError: 'Logistic Regression'

In [14]:
# Pick one student and explain their prediction
sample_idx = 0
sample = X_test.iloc[[sample_idx]]

prediction = best_model.predict(sample)[0]
print(f"Predicted at-risk: {prediction}")
print(f"Actual at-risk: {y_test.iloc[sample_idx]}")

# Force plot for this individual
if best_model_name in ['Random Forest', 'XGBoost']:
    shap.force_plot(explainer.expected_value, shap_values[sample_idx], sample, feature_names=feature_cols, matplotlib=True)
    

Predicted at-risk: 0
Actual at-risk: 0


In [15]:
import pickle

with open('../models/best_model.pkl', 'wb') as f:
    pickle.dump(best_model, f)

with open('../models/label_encoders.pkl', 'wb') as f:
    pickle.dump(le_dict, f)

with open('../models/feature_cols.pkl', 'wb') as f:
    pickle.dump(feature_cols, f)

print("Saved:", best_model_name)

Saved: Logisitic Regression
